# PixelClear Enhanced Framework
## CNN-Transformer Fusion Model with Multi-Degradation Handling

This notebook implements:
1. Lightweight Transformer attention for global context
2. CNN-Transformer fusion architecture
3. Compression artifact handling
4. Low-light enhancement module
5. Enhanced pixel-level explainability

**All code is self-contained - no external imports needed!**

In [9]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional, Dict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms.functional as TF
import math

BASE_DIR = Path("/Users/sachithwickramaseakara/Desktop/FYP/Implementation/PixelClear")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Base directory: {BASE_DIR}")
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

Base directory: /Users/sachithwickramaseakara/Desktop/FYP/Implementation/PixelClear
Device: cpu
PyTorch version: 2.9.1


## 1. CNN Components

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int = 1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, stride, padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return x

class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = self.sigmoid(avg_out + max_out)
        return x * out

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg_out, max_out], dim=1)
        attention = self.sigmoid(self.conv(combined))
        return x * attention

class CBAM(nn.Module):
    def __init__(self, channels: int, reduction: int = 16, kernel_size: int = 7):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x

class LightweightResidualBlock(nn.Module):
    def __init__(self, channels: int, use_attention: bool = True):
        super().__init__()
        self.use_attention = use_attention
        self.conv1 = DepthwiseSeparableConv(channels, channels)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = DepthwiseSeparableConv(channels, channels)
        if use_attention:
            self.attention = CBAM(channels, reduction=8)
        self.relu2 = nn.ReLU(inplace=True)
    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.relu1(out)
        out = self.conv2(out)
        if self.use_attention:
            out = self.attention(out)
        out = out + residual
        out = self.relu2(out)
        return out

class MultiScaleFeatureExtractor(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True)
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels // 4, out_channels // 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True)
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels // 4, out_channels // 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels // 4, out_channels // 4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels // 4),
            nn.ReLU(inplace=True)
        )
        self.branch4 = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels // 4, kernel_size=1, bias=False),
            # Note: No BatchNorm here because AdaptiveAvgPool2d(1) creates [B, C, 1, 1] which fails in training mode
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        b4 = F.interpolate(b4, size=x.shape[2:], mode='bilinear', align_corners=False)
        out = torch.cat([b1, b2, b3, b4], dim=1)
        return out

print("✓ CNN components defined")

✓ CNN components defined


## 2. Transformer Components

In [11]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size: int = 256, patch_size: int = 8, in_channels: int = 3, embed_dim: int = 64):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size, bias=False),
            nn.BatchNorm2d(embed_dim)
        )
    def forward(self, x):
        B, C, H, W = x.shape
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        return x

class LightweightSelfAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int = 4, qkv_bias: bool = False, attn_drop: float = 0.0, proj_drop: float = 0.0):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x, attn

class TransformerBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int = 4, mlp_ratio: float = 2.0, drop: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = LightweightSelfAttention(dim, num_heads=num_heads, attn_drop=drop, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim), nn.GELU(), nn.Dropout(drop),
            nn.Linear(mlp_hidden_dim, dim), nn.Dropout(drop)
        )
    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, attn_weights = self.attn(x_norm)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x, attn_weights

class GlobalContextTransformer(nn.Module):
    def __init__(self, in_channels: int = 32, embed_dim: int = 64, num_heads: int = 4, num_layers: int = 2, patch_size: int = 8, img_size: int = 256):
        super().__init__()
        self.patch_size = patch_size
        self.img_size = img_size
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = (img_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads=num_heads, mlp_ratio=2.0, drop=0.0) 
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.proj_back = nn.Sequential(nn.Linear(embed_dim, in_channels), nn.GELU())
    def forward(self, x, return_attention: bool = False):
        B, C, H, W = x.shape
        x_tokens = self.patch_embed(x)
        x_tokens = x_tokens + self.pos_embed
        attention_maps = []
        for block in self.blocks:
            x_tokens, attn = block(x_tokens)
            if return_attention:
                attention_maps.append(attn)
        x_tokens = self.norm(x_tokens)
        x_tokens = self.proj_back(x_tokens)
        h = w = int(math.sqrt(x_tokens.shape[1]))
        x_out = x_tokens.transpose(1, 2).reshape(B, C, h, w)
        if h != H or w != W:
            x_out = F.interpolate(x_out, size=(H, W), mode='bilinear', align_corners=False)
        if return_attention:
            return x_out, attention_maps
        return x_out

print("✓ Transformer components defined")

✓ Transformer components defined


## 3. Specialized Degradation Modules

In [12]:
class CompressionArtifactRemover(nn.Module):
    def __init__(self, channels: int = 32):
        super().__init__()
        self.block_detector = nn.Sequential(
            nn.Conv2d(channels, channels // 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels // 2), nn.ReLU(inplace=True),
            nn.Conv2d(channels // 2, channels, kernel_size=3, padding=1, bias=False), nn.Sigmoid()
        )
        self.smoother = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        )
    def forward(self, x):
        artifact_mask = self.block_detector(x)
        smoothed = self.smoother(x)
        out = x * (1 - artifact_mask) + smoothed * artifact_mask
        return out

class LowLightEnhancer(nn.Module):
    def __init__(self, channels: int = 32):
        super().__init__()
        self.illumination_estimator = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // 4, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 4, channels, kernel_size=1, bias=False), nn.Sigmoid()
        )
        self.enhancer = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        )
    def forward(self, x):
        illumination = self.illumination_estimator(x)
        enhanced = self.enhancer(x)
        out = x + enhanced * (1 - illumination)
        return out

print("✓ Specialized modules defined")

✓ Specialized modules defined


## 4. Fusion Architecture

In [13]:
class FusionBlock(nn.Module):
    def __init__(self, channels: int, use_transformer: bool = True, use_compression: bool = True, use_lowlight: bool = True):
        super().__init__()
        self.use_transformer = use_transformer
        self.use_compression = use_compression
        self.use_lowlight = use_lowlight
        self.cnn_block = LightweightResidualBlock(channels, use_attention=True)
        if use_transformer:
            self.transformer = GlobalContextTransformer(in_channels=channels, embed_dim=channels * 2, num_heads=4, num_layers=1, patch_size=8)
        if use_compression:
            self.compression_module = CompressionArtifactRemover(channels)
        if use_lowlight:
            self.lowlight_module = LowLightEnhancer(channels)
        self.fusion_gate = nn.Sequential(
            nn.Conv2d(channels * 2 if use_transformer else channels, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels), nn.Sigmoid()
        )
        self.output_conv = nn.Sequential(
            nn.Conv2d(channels * 2 if use_transformer else channels, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True)
        )
    def forward(self, x, return_attention: bool = False):
        cnn_feat = self.cnn_block(x)
        if self.use_compression:
            cnn_feat = self.compression_module(cnn_feat)
        if self.use_lowlight:
            cnn_feat = self.lowlight_module(cnn_feat)
        if self.use_transformer:
            transformer_feat, attention_maps = self.transformer(x, return_attention=True)
            combined = torch.cat([cnn_feat, transformer_feat], dim=1)
            fusion_weight = self.fusion_gate(combined)
            out = self.output_conv(combined)
            if return_attention:
                return out, attention_maps
            return out
        else:
            # When no transformer, return empty attention list if requested
            if return_attention:
                return cnn_feat, []
            return cnn_feat

class PixelClearFusionNet(nn.Module):
    def __init__(self, in_channels: int = 3, base_channels: int = 32, num_fusion_blocks: int = 4, use_transformer: bool = True, use_compression: bool = True, use_lowlight: bool = True, img_size: int = 256):
        super().__init__()
        self.use_transformer = use_transformer
        self.input_conv = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True)
        )
        self.multi_scale_extractor = MultiScaleFeatureExtractor(base_channels, base_channels)
        self.fusion_blocks = nn.ModuleList([
            FusionBlock(channels=base_channels, use_transformer=use_transformer and (i % 2 == 0), use_compression=use_compression, use_lowlight=use_lowlight)
            for i in range(num_fusion_blocks)
        ])
        self.output_conv = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, in_channels, kernel_size=3, padding=1)
        )
    def forward(self, x, return_features: bool = False, return_attention: bool = False):
        residual = x
        feat = self.input_conv(x)
        feat = self.multi_scale_extractor(feat)
        intermediate_features = [feat]
        all_attention_maps = []
        for block in self.fusion_blocks:
            if return_attention:
                feat, attn_maps = block(feat, return_attention=True)
                if attn_maps:  # Only extend if not empty
                    all_attention_maps.extend(attn_maps)
            else:
                feat = block(feat)
            intermediate_features.append(feat)
        out = self.output_conv(feat)
        out = out + residual
        if return_features and return_attention:
            return out, intermediate_features, all_attention_maps
        elif return_features:
            return out, intermediate_features
        elif return_attention:
            return out, all_attention_maps
        else:
            return out

print("✓ Fusion architecture defined")

✓ Fusion architecture defined


## 5. Enhanced Explainability Module

In [14]:
class EnhancedExplainabilityModule(nn.Module):
    def __init__(self):
        super().__init__()
    def generate_pixel_change_map(self, input_img: torch.Tensor, output_img: torch.Tensor) -> torch.Tensor:
        diff_r = torch.abs(output_img[:, 0:1] - input_img[:, 0:1])
        diff_g = torch.abs(output_img[:, 1:2] - input_img[:, 1:2])
        diff_b = torch.abs(output_img[:, 2:3] - input_img[:, 2:3])
        change_map = (diff_r + diff_g + diff_b) / 3.0
        change_map = (change_map - change_map.min()) / (change_map.max() - change_map.min() + 1e-8)
        return change_map
    def generate_improvement_map(self, input_img: torch.Tensor, output_img: torch.Tensor, target_img: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        def laplacian_variance(img):
            gray = 0.299 * img[:, 0] + 0.587 * img[:, 1] + 0.114 * img[:, 2]
            laplacian_kernel = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32, device=img.device)
            laplacian_kernel = laplacian_kernel.view(1, 1, 3, 3)
            gray_4d = gray.unsqueeze(1)
            laplacian = F.conv2d(gray_4d, laplacian_kernel, padding=1)
            return torch.var(laplacian, dim=[2, 3], keepdim=True)
        input_sharpness = laplacian_variance(input_img)
        output_sharpness = laplacian_variance(output_img)
        sharpness_improvement = (output_sharpness - input_sharpness) / (input_sharpness + 1e-8)
        input_brightness = torch.mean(input_img, dim=1, keepdim=True)
        output_brightness = torch.mean(output_img, dim=1, keepdim=True)
        brightness_improvement = output_brightness - input_brightness
        change_map = self.generate_pixel_change_map(input_img, output_img)
        results = {'change_map': change_map, 'sharpness_improvement': sharpness_improvement, 'brightness_improvement': brightness_improvement}
        if target_img is not None:
            error_map = torch.mean(torch.abs(output_img - target_img), dim=1, keepdim=True)
            results['error_map'] = error_map
        return results
    def generate_transformer_attention_visualization(self, attention_maps: List[torch.Tensor], img_size: int = 256, patch_size: int = 8) -> torch.Tensor:
        if not attention_maps:
            return None
        avg_attention = torch.stack([attn.mean(dim=1).mean(dim=1) for attn in attention_maps]).mean(dim=0)
        h = w = int(math.sqrt(avg_attention.shape[-1]))
        attention_spatial = avg_attention.reshape(-1, h, w)
        attention_spatial = attention_spatial.unsqueeze(1)
        attention_spatial = F.interpolate(attention_spatial, size=(img_size, img_size), mode='bilinear', align_corners=False)
        return attention_spatial.squeeze(1)
    def forward(self, input_img: torch.Tensor, output_img: torch.Tensor, features: Optional[List[torch.Tensor]] = None, attention_maps: Optional[List[torch.Tensor]] = None, target_img: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        results = self.generate_improvement_map(input_img, output_img, target_img)
        if attention_maps:
            transformer_attn = self.generate_transformer_attention_visualization(attention_maps, img_size=input_img.shape[-1])
            if transformer_attn is not None:
                results['transformer_attention'] = transformer_attn
        if features:
            feature_attention = torch.mean(features[-1], dim=1, keepdim=True)
            feature_attention = (feature_attention - feature_attention.min()) / (feature_attention.max() - feature_attention.min() + 1e-8)
            results['feature_attention'] = feature_attention
        return results

print("✓ Explainability module defined")

✓ Explainability module defined


## 6. Create and Test Model

In [15]:
# Create model with default configuration
model = PixelClearFusionNet(
    in_channels=3,
    base_channels=32,
    num_fusion_blocks=4,
    use_transformer=True,
    use_compression=True,
    use_lowlight=True,
    img_size=256
).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model created successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Model size: ~{total_params * 4 / (1024**2):.2f} MB")

Model created successfully!
Total parameters: 756,139
Model size: ~2.88 MB


In [16]:
# Test forward pass - FIXED VERSION
dummy_input = torch.randn(1, 3, 256, 256).to(DEVICE)

with torch.no_grad():
    # Test basic forward pass first
    output = model(dummy_input)
    print(f"✓ Basic forward pass successful!")
    print(f"Input shape: {dummy_input.shape}")
    print(f"Output shape: {output.shape}")
    
    # Test with features
    output, features = model(dummy_input, return_features=True)
    print(f"✓ Forward pass with features successful!")
    print(f"Number of feature maps: {len(features)}")
    
    # Test with attention (only if transformer is enabled)
    if model.use_transformer:
        output, attention_maps = model(dummy_input, return_attention=True)
        print(f"✓ Forward pass with attention successful!")
        print(f"Number of attention maps: {len(attention_maps)}")
    
    # Test with both
    output, features, attention_maps = model(dummy_input, return_features=True, return_attention=True)
    print(f"✓ Full forward pass successful!")
    print(f"Final output shape: {output.shape}")
    print(f"Feature maps: {len(features)}")
    print(f"Attention maps: {len(attention_maps)}")

✓ Basic forward pass successful!
Input shape: torch.Size([1, 3, 256, 256])
Output shape: torch.Size([1, 3, 256, 256])
✓ Forward pass with features successful!
Number of feature maps: 5
✓ Forward pass with attention successful!
Number of attention maps: 2
✓ Full forward pass successful!
Final output shape: torch.Size([1, 3, 256, 256])
Feature maps: 5
Attention maps: 2


## 7. Test Explainability Module

In [17]:
# Test explainability
explainer = EnhancedExplainabilityModule()

with torch.no_grad():
    output, features, attention_maps = model(dummy_input, return_features=True, return_attention=True)
    explain_results = explainer(dummy_input, output, features, attention_maps)

print("Explainability Maps Generated:")
for key, value in explain_results.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
print("✓ Explainability module working!")

Explainability Maps Generated:
  change_map: torch.Size([1, 1, 256, 256])
  sharpness_improvement: torch.Size([1, 1, 1, 1])
  brightness_improvement: torch.Size([1, 1, 256, 256])
  transformer_attention: torch.Size([1, 256, 256])
  feature_attention: torch.Size([1, 1, 256, 256])
✓ Explainability module working!
